In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# تقدير طاقة الحالة الأساسية لسلسلة هايزنبرغ باستخدام VQE

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*تقدير الاستخدام: 37 دقيقة على معالج Heron (ملاحظة: هذا تقدير فحسب. قد يختلف وقت التشغيل الفعلي.)*
## نتائج التعلم
بعد إتمام هذا البرنامج التعليمي، يمكنك توقع فهم المعلومات التالية:
- كيفية نمذجة سلسلة سبين هايزنبرغ كهاميلتونيان كمومي باستخدام Qiskit
- كيفية استخدام محسِّن SPSA لتقدير طاقة الحالة الأساسية لنظام كمومي
- كيفية تنفيذ سير العمل التقريبية على عتاد IBM&reg; الكمومي باستخدام أدوات Qiskit Runtime الأولية والجلسات

## المتطلبات الأساسية
يُوصى بالتعرف على هذه الموضوعات:
- [أساسيات المعلومات الكمومية](/learning/courses/basics-of-quantum-information)
- [مقدمة إلى أنماط Qiskit](/guides/intro-to-patterns)
- [تصميم الخوارزمية التقريبية](/learning/courses/variational-algorithm-design)

## الخلفية
سلسلة سبين هايزنبرغ هي أحد أكثر النماذج التي تمت دراستها على نطاق واسع في فيزياء المادة المكثفة والمغناطيسية الكمومية. تصف شبكةً أحادية البُعد من السبينات الكمومية المتفاعلة، حيث تترابط السبينات المجاورة عبر تفاعلات التبادل. يُعطى الهاميلتونيان لنموذج هايزنبرغ متماثل التوجه مع مجال مغناطيسي خارجي بالمعادلة:

$$H = \sum_{\langle i,j \rangle} \left( J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right) + \sum_{i} h_i Z_i,$$

حيث $X_i$ و$Y_i$ و$Z_i$ هي عوامل باولي التي تعمل على الموقع $i$، والمجموع $\langle i,j \rangle$ يجري على أزواج الجيران الأقرب، و$J_x = J_y = J_z = 0.5$ هي ثوابت اقتران التبادل (متماثلة في هذا البرنامج التعليمي)، و$h_i$ يمثل مجالاً مغناطيسياً خارجياً يعتمد على الموقع. في هذا البرنامج التعليمي، تُسحب قيم المجال المغناطيسي عشوائياً من الفترة $[-1, 1]$. لاحظ أنه في التنفيذ أدناه، تُحدَّد مجموعة أزواج "الجيران الأقرب" بواسطة اقتران المعالج الخلفي الأصلي بين أول $N$ كيوبت، والتي قد لا تُشكِّل سلسلة خطية صارمة بحسب طبولوجيا الجهاز.

إن فهم طاقة الحالة الأساسية لهذا الهاميلتونيان أمر بالغ الأهمية في الفيزياء. تُرمِّز الحالة الأساسية معلومات عن الانتقالات الطورية الكمومية وبنية التشابك والترتيب المغناطيسي. كلاسيكياً، يصبح حساب طاقة الحالة الأساسية الدقيقة أمراً غير قابل للمعالجة مع نمو عدد السبينات، إذ يتوسع بُعد فضاء هيلبرت أسياً بمقدار $2^N$ لـ $N$ سبين. وهذا ما يجعله مرشحاً طبيعياً للمحاكاة الكمومية.

المحلِّل الكمومي التقريبي (VQE) هو خوارزمية هجينة كمومية-كلاسيكية مصممة لتقدير طاقة الحالة الأساسية لهاميلتونيان. يعمل عن طريق تهيئة حالة كمومية مُعلمَّنة $|\psi(\theta)\rangle$ (تُعرف بـ ansatz) على حاسوب كمومي وقياس القيمة المتوقعة $\langle \psi(\theta) | H | \psi(\theta) \rangle$. ثم يضبط محسِّن كلاسيكي المعاملات $\theta$ بصورة تكرارية لتقليل هذه الطاقة، مستفيداً من مبدأ التقريب الذي يضمن أن الطاقة المقاسة هي دائماً حدٌّ أعلى لطاقة الحالة الأساسية الفعلية.

في هذا البرنامج التعليمي، نستخدم ansatz من نوع `efficient_su2` من مكتبة الدوائر في Qiskit، والتي تبني طبقات من التدويرات أحادية الكيوبت وبوابات التشابك. يُنفَّذ التحسين باستخدام خوارزمية التقريب العشوائي للاضطراب المتزامن (SPSA)، المناسبة للعتاد الكمومي الضوضائي لأنها تقدِّر التدرجات باستخدام تقييمَي دالة فقط لكل تكرار بصرف النظر عن عدد المعاملات.
## المتطلبات
قبل البدء في هذا البرنامج التعليمي، تأكد من تثبيت ما يلي:

* Qiskit SDK الإصدار 2.0 أو أحدث، مع دعم [التصور المرئي](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime الإصدار 0.44 أو أحدث (`pip install qiskit-ibm-runtime`)
## الإعداد

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## مثال صغير النطاق

في هذا القسم، نتناول كل خطوة من خطوات نمط Qiskit على نطاق صغير، مع شرح المكونات الرئيسية أثناء بناء سير العمل.

### الخطوة 1: تحويل المدخلات الكلاسيكية إلى مسألة كمومية

* المدخل: عدد السبينات (spins)
* المخرج: دالة الاختبار (Ansatz) وهاميلتونيان يُنمذجان سلسلة هايزنبرغ

نبني دالة الاختبار (Ansatz) والهاميلتونيان اللذين ينمذجان سلسلة هايزنبرغ المكونة من 10 سبينات. في هذه الخطوة، سنبني هاميلتونيان هايزنبرغ لـ 10 سبينات على خريطة اقتران المعالج الخلفي الأقل انشغالاً ونُهيئ ansatz من نوع `efficient_su2`.

In [2]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

### الخطوة 2: تحسين المسألة لتنفيذها على العتاد الكمومي
* المدخل: دائرة مجردة، ومتغير مرصود (observable)
* المخرج: الدائرة المستهدفة والمتغير المرصود، محسَّنَين للمعالج الكمومي (QPU) المختار

نستخدم دالة `generate_preset_pass_manager` من Qiskit لتوليد روتين تحسين آلي لدائرتنا بالنسبة إلى المعالج الكمومي المختار. نختار `optimization_level=3` الذي يوفر أعلى مستوى من التحسين بين مديري التمريرات المُعدَّة مسبقاً. كما نُدرج تمريرات الجدولة `ALAPScheduleAnalysis` و`PadDynamicalDecoupling` لقمع أخطاء إزالة الترابط الكمومي.

In [3]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

### الخطوة 3: التنفيذ باستخدام Qiskit primitives
* المدخل: الدائرة المستهدفة والمتغير المرصود
* المخرج: نتائج عملية التحسين

نُصغِّر طاقة الحالة الأساسية المُقدَّرة للنظام عبر تحسين معاملات الدائرة. نستخدم الأداة الأولية `Estimator` من Qiskit Runtime لتقييم دالة التكلفة خلال عملية التحسين.

بما أننا حسَّنَّا الدائرة للمعالج الخلفي في الخطوة 2، يمكننا تجنب إجراء التصميم المادي (transpilation) على خادم Runtime عبر تعيين `skip_transpilation=True` وتمرير الدائرة المحسَّنة. في هذا العرض التوضيحي، سنُشغِّل النمط على معالج كمومي (QPU) باستخدام أدوات `qiskit-ibm-runtime` الأولية. للتشغيل باستخدام الأدوات الأولية المبنية على متجه الحالة في `qiskit`، استبدل كتلة الكود المستخدمة لأدوات Qiskit Runtime الأولية بالكتلة المُعلَّقة.

في هذا البرنامج التعليمي، نستخدم التقريب العشوائي للاضطراب المتزامن (SPSA)، وهو محسِّن قائم على التدرج. سنقدم له مقدمة موجزة، ثم نوفر الكود لتنفيذ SPSA باستخدام Qiskit v2.0.
### تعريف بـ SPSA
التقريب العشوائي للاضطراب المتزامن (SPSA) [\[1\]](#references) هو خوارزمية تحسين تقرِّب متجه التدرج بالكامل باستخدام استدعاءَي دالة فقط في كل تكرار. لتكن $f:\mathbb{R}^p\rightarrow \mathbb{R}$ دالة التكلفة ذات $p$ معاملاً مراد تحسينها، وليكن $x_i\in \mathbb{R}^p$ متجه المعاملات عند الخطوة $i^{th}$ من التكرار. لحساب التدرج، يُنشأ متجه عشوائي $\Delta_i$ بحجم $p$، حيث يُسحب كل عنصر $\Delta_{ij}$، $\forall$ $j\in {1,2,...,p}$، بشكل منتظم من ${-1, 1}$. بعد ذلك، يُضرب كل عنصر من المتجه العشوائي $\Delta_i$ في قيمة صغيرة $c_i$ لإنشاء اضطراب عشوائي. يُقدَّر التدرج حينئذٍ بالمعادلة:

$$[\nabla f(x_i)]_j \approx \frac{f(x_i + c_i \Delta_i) - f(x_i - c_i \Delta_i)}{2c_i\Delta_{ij}}.$$

بالحدس، بما أنه يُطبَّق اضطراب عشوائي أثناء تقدير التدرج، يُتوقَّع أن تُحتمَل الانحرافات الصغيرة في القيم الدقيقة لـ $f$ الناتجة عن الضوضاء وتُحسَب حسابها. في الواقع، يشتهر SPSA بمتانته أمام الضوضاء، ويتطلب استدعاءَي عتاد فقط في كل تكرار. لذا، يُعدّ من المحسِّنات الأكثر تفضيلاً لتنفيذ الخوارزميات التقريبية.

في هذا البرنامج التعليمي، تُحسَب المعاملات الفائقة للتكرار $i^{th}$، $a_i$ و $c_i$، وفق

$$a_i = \frac{a}{(A + i + 1)^\alpha} \quad \text{and} \quad c_i = \frac{c}{(i+1)^\gamma},$$

حيث تُؤخذ القيم الثابتة على أنها $A = 30$، $\alpha = 0.9$، $a = 0.3$، $c = 0.1$، و $\gamma = 0.4$. هذه القيم مأخوذة من [\[2\]](#references). يُعدّ الضبط المناسب للمعاملات الفائقة ضرورياً لاستخراج أداء جيد من SPSA.

In [4]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [8]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        estimator.options.environment.job_tags = ["TUT_HSVQE"]
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [9]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

هنا نضع `maxiter = 50`. لاحظ أنه بما أن كل تكرار يتطلب استدعاءَي للدالة لحساب التدرج، فإن إجمالي عدد استدعاءات الدالة سيكون $2 \times \text{maxiter}$. يمكن زيادة `maxiter` إلى أي قيمة أعلى للحصول على تقدير طاقة أفضل.

In [10]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

Fx Iters. done: 101 [Current cost: -3.03843]

### Step 4: Post-process and return result in desired classical format

* Input: Ground-state energy estimates during optimization
* Output: Estimated ground-state energy

In [11]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-3.03842968]


In [12]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

A large-scale hardware example is not included in this tutorial. As the number of qubits increases, VQE encounters significant challenges due to the [barren plateau](/learning/courses/variational-algorithm-design/optimization-loops#barren-plateaus) phenomenon: the gradient of the cost function vanishes exponentially with system size, making optimization practically infeasible for large circuits. Combined with hardware noise, this means that scaling VQE to larger spin chains does not produce reliably reproducible results. For approaches that overcome these limitations, see the Next Steps section below.

## Challenge

Now that you have a working VQE implementation for the Heisenberg chain, try the following:

1. **Experiment with ansatz depth:** Modify the `reps` parameter in `efficient_su2` (for example, try `reps=1` and `reps=3`). How does ansatz depth affect the estimated ground-state energy and convergence speed? At what point do you observe diminishing returns or instability?
2. **Tune SPSA hyperparameters:** Adjust the learning rate schedule parameters (`a`, `c`, `alpha`, `gamma`, `A`) and observe how they impact convergence. Can you find a configuration that converges faster than the defaults used here?
3. **Compare coupling topologies:** Instead of using the backend's native coupling map, try constructing a simple nearest-neighbor linear chain and compare the results. How does the connectivity of the physical hardware affect the transpiled circuit depth and final energy estimate?

## References

\[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

\[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
  If you found this work interesting, you might be interested in the following material:

  * **Try Sample-based Quantum Diagonalization (SQD):** As demonstrated in this tutorial, VQE faces challenges at scale due to barren plateaus and high measurement overhead. IBM has developed [Sample-based Quantum Diagonalization (SQD)](https://www.ibm.com/quantum/blog/quantum-diagonalization) as a more scalable alternative. Unlike VQE, SQD avoids variational optimization entirely; instead, a quantum computer generates samples and a classical computer projects the Hamiltonian onto a subspace spanned by those samples and diagonalizes it. This provides an upper bound to the ground-state energy with significantly fewer measurements and without susceptibility to barren plateaus. Follow the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization) to see this approach in action.
  * **Explore the Quantum Diagonalization Algorithms course:** Deepen your understanding of both VQE and SQD, including their trade-offs, in the [Quantum diagonalization algorithms](/learning/courses/quantum-diagonalization-algorithms) course on IBM Quantum Learning.
</Admonition>